# Chess Position Evaluation with Graph Neural Networks

A ResGATv2 model that evaluates chess positions by representing the board as a graph:
- **64 nodes** (one per square) with 21-dimensional features (piece one-hot, position, castling rights, etc.)
- **Edges** from legal moves with 10-dimensional features (capture, promotion, displacement, promotion type, etc.)
- **Dual head**: WDL value classification + per-edge policy for move selection
- **WDL classification** (Win/Draw/Loss) instead of raw centipawn regression for stable training
- **Self-play training** via MCTS with neural network guidance (AlphaZero-style)

Based on research from AlphaGateau (NeurIPS 2024), Alwer & Plaat (2023), and Czech et al. (2023).

In [ ]:
!pip install -q torch torch_geometric chess pandas numpy scikit-learn matplotlib tqdm

In [ ]:
# ============================================================
# Configuration — edit these to control the run
# ============================================================

# Path to CSV dataset (Kaggle default)
DATA_PATH = "/kaggle/input/stockfish-best-moves-compilation/dataset_eval.csv"

# How many positions to use (None = full dataset)
NUM_SAMPLES = None

# Training hyperparameters
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3
PATIENCE = 20
WARMUP_EPOCHS = 5

# Model architecture
HIDDEN = 128
HEADS = 4
NUM_BLOCKS = 4
DROPOUT = 0.2
EDGE_DIM = 10  # 10d edge features (with promotion type one-hot)

# Caching (saves processed graphs to disk so reruns are fast)
USE_CACHE = True
CACHE_PATH = f"processed_graphs_{NUM_SAMPLES}.pt"

# Output paths
BEST_MODEL_PATH = "best_model.pt"

# ── Self-play configuration ──
RUN_SELFPLAY = True   # Set to True to run self-play after supervised training

SELFPLAY_ITERATIONS = 50        # More iterations since each is now fast
SELFPLAY_GAMES_PER_ITER = 10    # Fewer games per iter (was 30 — too slow)
SELFPLAY_SIMULATIONS = 16       # 16 sims is enough for training signal (was 64 — main bottleneck)
SELFPLAY_MAX_MOVES = 200        # Cap game length (was 512 — draws ran forever)
SELFPLAY_BUFFER_SIZE = 100_000
SELFPLAY_LR = 1e-4
SELFPLAY_TRAIN_STEPS = 50       # Scale with buffer, don't overtrain on few positions
SELFPLAY_EVAL_INTERVAL = 10     # Eval less often (evals are expensive)
SELFPLAY_EVAL_GAMES = 10        # Fewer eval games (was 20)
SELFPLAY_EVAL_SIMS = 16         # Sims for eval games

# Policy pretraining (bootstrap before self-play)
PRETRAIN_POLICY = True   # Pre-train policy head on Stockfish best moves
PRETRAIN_EPOCHS = 10
PRETRAIN_SAMPLES = None  # None = use full dataset

print("Configuration loaded.")

In [ ]:
# ============================================================
# Imports and device setup
# ============================================================

import re
import math
import copy
import json
import time
import random
import logging
from collections import deque
from pathlib import Path

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, BatchNorm
from tqdm.auto import tqdm

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ChessGCN")

# ── Device ──
if torch.cuda.is_available():
    device = torch.device("cuda")
    log.info(f"Using CUDA — {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    log.info("Using MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    log.info("Using CPU")

log.info(f"PyTorch {torch.__version__}")

## 1. Data Pipeline

**Evaluation parsing** → **WDL conversion** → **FEN to graph** → **Stratified sampling** → **Caching**

In [ ]:
# ============================================================
# Data pipeline: FEN → Graph with WDL labels
# ============================================================

# Stockfish WDL calibration constant
WDL_K = 111.0

PROMO_MAP = {chess.QUEEN: 0, chess.ROOK: 1, chess.BISHOP: 2, chess.KNIGHT: 3}


def parse_evaluation(eval_str):
    """Convert evaluation string to centipawns. Mate → ±10000 scaled by distance."""
    mate_match = re.match(r"M(-?\d+)", str(eval_str))
    if mate_match:
        mate_moves = int(mate_match.group(1))
        if mate_moves > 0:
            return 10000 - (mate_moves - 1) * 10
        else:
            return -10000 + (abs(mate_moves) - 1) * 10
    return float(eval_str)


def cp_to_wdl(cp):
    """Convert centipawn evaluation to (win, draw, loss) probabilities."""
    if cp >= 9000:
        return (1.0, 0.0, 0.0)
    if cp <= -9000:
        return (0.0, 0.0, 1.0)
    win = 1.0 / (1.0 + math.exp(-cp / WDL_K))
    loss = 1.0 / (1.0 + math.exp(cp / WDL_K))
    draw = max(1.0 - win - loss, 0.0)
    total = win + draw + loss
    return (win / total, draw / total, loss / total)


def fen_to_graph(fen, wdl=None):
    """
    Convert a FEN string to a PyG Data object.

    Node features (21d per square):
      - Piece one-hot (12d): WP,WN,WB,WR,WQ,WK,BP,BN,BB,BR,BQ,BK
      - Empty flag (1d)
      - File (1d, normalized), Rank (1d, normalized)
      - Side to move (1d), Castling rights (4d), Half-move clock (1d)

    Edge features (10d per legal move):
      - is_capture, is_promotion, is_castling, is_en_passant, dx, dy
      - promotion type one-hot (4d: queen, rook, bishop, knight)
    """
    board = chess.Board(fen)

    side_to_move = 1.0 if board.turn == chess.WHITE else 0.0
    castling = [
        float(board.has_kingside_castling_rights(chess.WHITE)),
        float(board.has_queenside_castling_rights(chess.WHITE)),
        float(board.has_kingside_castling_rights(chess.BLACK)),
        float(board.has_queenside_castling_rights(chess.BLACK)),
    ]
    halfmove = min(board.halfmove_clock, 100) / 100.0
    PIECE_OFFSET = {chess.WHITE: 0, chess.BLACK: 6}

    node_features = []
    for square in chess.SQUARES:
        feat = [0.0] * 21
        piece = board.piece_at(square)
        if piece:
            idx = PIECE_OFFSET[piece.color] + (piece.piece_type - 1)
            feat[idx] = 1.0
        else:
            feat[12] = 1.0
        feat[13] = chess.square_file(square) / 7.0
        feat[14] = chess.square_rank(square) / 7.0
        feat[15] = side_to_move
        feat[16] = castling[0]
        feat[17] = castling[1]
        feat[18] = castling[2]
        feat[19] = castling[3]
        feat[20] = halfmove
        node_features.append(feat)

    src, dst, edge_attrs, move_uci = [], [], [], []
    for move in board.legal_moves:
        from_sq, to_sq = move.from_square, move.to_square
        src.append(from_sq)
        dst.append(to_sq)
        move_uci.append(move.uci())
        dx = (chess.square_file(to_sq) - chess.square_file(from_sq)) / 7.0
        dy = (chess.square_rank(to_sq) - chess.square_rank(from_sq)) / 7.0
        promo = [0.0, 0.0, 0.0, 0.0]
        if move.promotion is not None:
            promo[PROMO_MAP[move.promotion]] = 1.0
        edge_attrs.append([
            float(board.is_capture(move)),
            float(move.promotion is not None),
            float(board.is_castling(move)),
            float(board.is_en_passant(move)),
            dx, dy,
            *promo,
        ])

    if len(src) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data.move_uci = move_uci
    if wdl is not None:
        data.y = torch.tensor(wdl, dtype=torch.float)
    return data


def load_and_sample(df, num_samples, random_state=42):
    """Stratified sampling by evaluation bins."""
    bins = [-float("inf"), -5000, -2000, -1000, -500, -200, -100, -50,
            0, 50, 100, 200, 500, 1000, 2000, 5000, float("inf")]
    df["bin"] = pd.cut(df["cp"], bins=bins, labels=False)
    sampled = df.groupby("bin", group_keys=False).apply(
        lambda g: g.sample(
            n=min(len(g), max(1, int(num_samples * len(g) / len(df)))),
            random_state=random_state,
        ),
        include_groups=False,
    )
    sampled = df.loc[sampled.index]
    remaining = num_samples - len(sampled)
    if remaining > 0:
        leftover = df.drop(sampled.index)
        extra = leftover.sample(n=min(remaining, len(leftover)), random_state=random_state)
        sampled = pd.concat([sampled, extra])
    result = sampled.head(num_samples).reset_index(drop=True)
    if "bin" in result.columns:
        result = result.drop(columns=["bin"])
    if "bin" in df.columns:
        df.drop(columns=["bin"], inplace=True)
    return result


log.info("Data functions defined.")

In [ ]:
# ============================================================
# Load dataset + stratified sampling + build graphs
# ============================================================

t0 = time.time()
log.info(f"Loading CSV from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
log.info(f"  Total rows: {len(df):,}")
log.info(f"  Columns: {list(df.columns)}")

# Parse evaluations to centipawns
df["cp"] = df["Evaluation"].apply(parse_evaluation)
log.info(f"  Eval range: [{df['cp'].min():.0f}, {df['cp'].max():.0f}] cp")
log.info(f"  Eval mean: {df['cp'].mean():.1f}, std: {df['cp'].std():.1f}")

# ── Stratified sampling ──
if NUM_SAMPLES is not None and NUM_SAMPLES < len(df):
    log.info(f"Stratified sampling {NUM_SAMPLES:,} positions...")
    df = load_and_sample(df, NUM_SAMPLES)
    log.info(f"  Sampled {len(df):,} positions")
else:
    log.info("Using full dataset (no sampling)")

log.info(f"  CSV loading took {time.time() - t0:.1f}s")

# ── Build graphs (with caching) ──
cache_file = Path(CACHE_PATH) if USE_CACHE else None

if cache_file and cache_file.exists():
    log.info(f"Loading cached graphs from {cache_file}...")
    graphs = torch.load(cache_file, weights_only=False)
    log.info(f"  Loaded {len(graphs):,} cached graphs")
else:
    log.info("Building graphs from FEN positions...")
    t1 = time.time()
    graphs = []
    errors = 0
    for i, row in tqdm(df.iterrows(), total=len(df), desc="FEN → Graph"):
        fen = row["Position"]
        cp = row["cp"]
        wdl = cp_to_wdl(cp)
        try:
            graph = fen_to_graph(fen, wdl)
            graph.cp = torch.tensor([cp], dtype=torch.float)
            graphs.append(graph)
        except Exception as e:
            errors += 1
            if errors <= 5:
                log.warning(f"Error at index {i}, FEN='{fen}': {e}")

    elapsed = time.time() - t1
    log.info(f"  Built {len(graphs):,} graphs in {elapsed:.1f}s ({len(graphs)/elapsed:.0f} graphs/s)")
    if errors > 0:
        log.warning(f"  {errors} errors during graph construction")

    if cache_file:
        torch.save(graphs, cache_file)
        log.info(f"  Cached to {cache_file}")

# ── Quick sanity check ──
g = graphs[0]
log.info(f"Sample graph: {g.x.shape[0]} nodes, {g.edge_index.shape[1]} edges, "
         f"node_dim={g.x.shape[1]}, edge_dim={g.edge_attr.shape[1]}, "
         f"WDL target={[f'{v:.2f}' for v in g.y.tolist()]}")

In [ ]:
# ============================================================
# Train / Val / Test split + DataLoaders
# ============================================================

train_graphs, temp_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
val_graphs, test_graphs = train_test_split(temp_graphs, test_size=0.5, random_state=42)

log.info(f"Split: Train={len(train_graphs):,}  Val={len(val_graphs):,}  Test={len(test_graphs):,}")

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)

log.info(f"Batches per epoch: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

# ── Plot WDL distribution ──
train_wdl_classes = [g.y.argmax().item() for g in train_graphs]
class_names = ["Win", "Draw", "Loss"]
counts = [train_wdl_classes.count(i) for i in range(3)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_names, counts, color=["#2e7d32", "#757575", "#c62828"])
ax.set_ylabel("Count")
ax.set_title("Training Set WDL Class Distribution")
for i, c in enumerate(counts):
    ax.text(i, c + len(train_graphs)*0.01, f"{c:,}\n({c/len(train_graphs)*100:.1f}%)",
            ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Model Architecture

**ResGATv2**: GATv2Conv with edge features, residual connections, batch normalization.

In [ ]:
# ============================================================
# Model: ResGATv2 with dual head (value + policy)
# ============================================================

class ResGATv2Block(nn.Module):
    """Residual block with two GATv2Conv layers."""
    def __init__(self, channels, heads, edge_dim, dropout=0.1):
        super().__init__()
        self.conv1 = GATv2Conv(
            channels, channels // heads, heads=heads,
            edge_dim=edge_dim, concat=True, dropout=dropout,
        )
        self.bn1 = BatchNorm(channels)
        self.conv2 = GATv2Conv(
            channels, channels // heads, heads=heads,
            edge_dim=edge_dim, concat=True, dropout=dropout,
        )
        self.bn2 = BatchNorm(channels)

    def forward(self, x, edge_index, edge_attr):
        residual = x
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = self.bn2(self.conv2(x, edge_index, edge_attr))
        x = F.relu(x + residual)
        return x


class ChessGATv2(nn.Module):
    """
    Graph Attention Network for chess position evaluation.

    Dual head:
      - Value head: global mean pool → FC → 3-class WDL
      - Policy head: per-edge MLP(h_src, h_dst, edge_attr) → 1 logit per move
    """
    def __init__(self, node_dim=21, edge_dim=EDGE_DIM, hidden=128, heads=4,
                 num_blocks=4, dropout=0.2, policy_head=True):
        super().__init__()
        self.has_policy_head = policy_head
        self.input_proj = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU())
        self.blocks = nn.ModuleList([
            ResGATv2Block(hidden, heads, edge_dim, dropout=0.1)
            for _ in range(num_blocks)
        ])
        self.value_head = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 3),  # Win, Draw, Loss
        )
        if policy_head:
            self.policy_mlp = nn.Sequential(
                nn.Linear(hidden * 2 + edge_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 1),
            )

    def forward(self, data):
        x, edge_index, edge_attr, batch = (
            data.x, data.edge_index, data.edge_attr, data.batch
        )
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, edge_index, edge_attr)

        # Value head
        x_pool = global_mean_pool(x, batch)
        value_logits = self.value_head(x_pool)

        # Policy head
        policy_logits = None
        if self.has_policy_head and edge_index.size(1) > 0:
            src_emb = x[edge_index[0]]
            dst_emb = x[edge_index[1]]
            edge_input = torch.cat([src_emb, dst_emb, edge_attr], dim=1)
            policy_logits = self.policy_mlp(edge_input).squeeze(-1)

        return value_logits, policy_logits


# ── Instantiate ──
model = ChessGATv2(
    hidden=HIDDEN, heads=HEADS, num_blocks=NUM_BLOCKS, dropout=DROPOUT,
    policy_head=True,
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log.info(f"Model created: {num_params:,} trainable parameters")
log.info(f"Architecture: {NUM_BLOCKS} ResGATv2 blocks, {HEADS} heads, hidden={HIDDEN}")
log.info(f"Dual head: value (WDL) + policy (per-edge)")
print(model)

## 3. Training

In [ ]:
# ============================================================
# Training and evaluation functions
# ============================================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits, _ = model(batch)
        target = batch.y.view(-1, 3)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.kl_div(log_probs, target, reduction="batchmean")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * target.size(0)
        correct += (logits.argmax(1) == target.argmax(1)).sum().item()
        total += target.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets, all_cp_true, all_values_pred = [], [], [], []

    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        target = batch.y.view(-1, 3)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.kl_div(log_probs, target, reduction="batchmean")
        total_loss += loss.item() * target.size(0)

        probs = F.softmax(logits, dim=1)
        pred_class = probs.argmax(dim=1)
        true_class = target.argmax(dim=1)
        correct += (pred_class == true_class).sum().item()
        total += target.size(0)

        value_pred = probs[:, 0] - probs[:, 2]
        all_preds.extend(pred_class.cpu().tolist())
        all_targets.extend(true_class.cpu().tolist())
        all_values_pred.extend(value_pred.cpu().tolist())
        if hasattr(batch, "cp"):
            all_cp_true.extend(batch.cp.view(-1).cpu().tolist())

    avg_loss = total_loss / total
    accuracy = correct / total

    # Value MAE and correlation
    values_pred_t = torch.tensor(all_values_pred)
    value_mae, correlation = None, None
    if all_cp_true:
        values_true = torch.tensor([
            2.0 / (1.0 + math.exp(-cp / WDL_K)) - 1.0 for cp in all_cp_true
        ])
        value_mae = (values_pred_t - values_true).abs().mean().item()
        if len(values_true) > 1:
            vp = values_pred_t - values_pred_t.mean()
            vt = values_true - values_true.mean()
            correlation = ((vp * vt).sum() / (vp.norm() * vt.norm() + 1e-8)).item()

    return {
        "loss": avg_loss, "accuracy": accuracy,
        "value_mae": value_mae, "correlation": correlation,
        "preds": all_preds, "targets": all_targets,
        "cp_true": all_cp_true, "values_pred": all_values_pred,
    }


# ── LR schedule: linear warmup + cosine decay ──
def get_scheduler(optimizer, warmup, total):
    def lr_lambda(epoch):
        if epoch < warmup:
            return epoch / max(warmup, 1)
        progress = (epoch - warmup) / max(total - warmup, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


log.info("Training functions defined.")

In [ ]:
# ============================================================
# Training loop
# ============================================================

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = get_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_corr": []}
best_val_loss = float("inf")
patience_counter = 0
train_start = time.time()

log.info(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LR}, patience={PATIENCE}")
log.info(f"{'Epoch':>5} | {'Train Loss':>10} {'Acc':>6} | {'Val Loss':>10} {'Acc':>6} | {'Corr':>7} {'MAE':>7} | {'LR':>9} | {'Time':>6}")
log.info("-" * 90)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    val_result = evaluate(model, val_loader, device)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_result["loss"])
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_result["accuracy"])
    history["val_corr"].append(val_result["correlation"])

    corr_s = f"{val_result['correlation']:.4f}" if val_result["correlation"] else "  N/A  "
    mae_s = f"{val_result['value_mae']:.4f}" if val_result["value_mae"] else "  N/A  "

    marker = ""
    if val_result["loss"] < best_val_loss:
        best_val_loss = val_result["loss"]
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = " *"
    else:
        patience_counter += 1

    log.info(
        f"{epoch:5d} | {train_loss:10.4f} {train_acc:5.1%} | "
        f"{val_result['loss']:10.4f} {val_result['accuracy']:5.1%} | "
        f"{corr_s} {mae_s} | {lr:9.2e} | {epoch_time:5.1f}s{marker}"
    )

    if patience_counter >= PATIENCE:
        log.info(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

total_time = time.time() - train_start
log.info(f"Training complete in {total_time/60:.1f} minutes")
log.info(f"Best validation loss: {best_val_loss:.4f}")

## 4. Evaluation

In [ ]:
# ============================================================
# Training curves
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="Train", linewidth=1.5)
axes[0].plot(history["val_loss"], label="Val", linewidth=1.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("KL-Div Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_acc"], label="Train", linewidth=1.5)
axes[1].plot(history["val_acc"], label="Val", linewidth=1.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("WDL Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

valid_corr = [c for c in history["val_corr"] if c is not None]
if valid_corr:
    axes[2].plot(valid_corr, label="Val Correlation", color="green", linewidth=1.5)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Pearson r")
    axes[2].set_title("Value Correlation")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
log.info("Training curves plotted.")

In [ ]:
# ============================================================
# Final test set evaluation
# ============================================================

model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True, map_location=device))
test_result = evaluate(model, test_loader, device)

log.info("=" * 50)
log.info("FINAL TEST SET RESULTS")
log.info("=" * 50)
log.info(f"  Test Loss:       {test_result['loss']:.4f}")
log.info(f"  Test WDL Acc:    {test_result['accuracy']:.1%}")
if test_result["value_mae"] is not None:
    log.info(f"  Value MAE:       {test_result['value_mae']:.4f}")
if test_result["correlation"] is not None:
    log.info(f"  Correlation:     {test_result['correlation']:.4f}")

# ── Per-class accuracy ──
from collections import Counter
class_correct = Counter()
class_total = Counter()
for p, t in zip(test_result["preds"], test_result["targets"]):
    class_total[t] += 1
    if p == t:
        class_correct[t] += 1

for cls_id, cls_name in enumerate(["Win", "Draw", "Loss"]):
    total = class_total.get(cls_id, 0)
    correct = class_correct.get(cls_id, 0)
    acc = correct / total if total > 0 else 0
    log.info(f"  {cls_name:>4s} accuracy: {acc:.1%} ({correct}/{total})")

In [ ]:
# ============================================================
# Confusion matrix
# ============================================================

cm = confusion_matrix(test_result["targets"], test_result["preds"], labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=["Win", "Draw", "Loss"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap="Blues", ax=ax)
ax.set_title("WDL Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
log.info("Confusion matrix plotted.")

In [ ]:
# ============================================================
# Predicted vs true evaluation scatter plot
# ============================================================

if test_result["cp_true"]:
    cp_pred = []
    for v in test_result["values_pred"]:
        v_clamped = max(-0.999, min(0.999, v))
        p_win = (v_clamped + 1.0) / 2.0
        p_loss = 1.0 - p_win
        if p_loss < 1e-6:
            cp_pred.append(5000)
        elif p_win < 1e-6:
            cp_pred.append(-5000)
        else:
            cp_pred.append(WDL_K * math.log(p_win / p_loss))

    cp_true_clamped = [max(-5000, min(5000, c)) for c in test_result["cp_true"]]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(cp_true_clamped, cp_pred, alpha=0.05, s=2)
    ax.plot([-5000, 5000], [-5000, 5000], "r--", linewidth=1, label="Perfect")
    ax.set_xlabel("True Evaluation (cp)")
    ax.set_ylabel("Predicted Evaluation (cp)")
    ax.set_title("Predicted vs True Evaluation")
    ax.legend()
    ax.set_xlim(-5000, 5000)
    ax.set_ylim(-5000, 5000)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig("eval_scatter.png", dpi=150)
    plt.show()
    log.info("Scatter plot generated.")
else:
    log.warning("No centipawn data available for scatter plot.")

In [ ]:
# ============================================================
# Sample predictions — inspect a few positions
# ============================================================

log.info("Sample predictions from test set:")
log.info(f"{'FEN (truncated)':>45s} | {'True CP':>8s} | {'True WDL':>12s} | {'Pred WDL':>12s} | {'Pred Val':>8s}")
log.info("-" * 100)

sample_indices = np.random.RandomState(42).choice(len(test_graphs), size=min(10, len(test_graphs)), replace=False)
for idx in sample_indices:
    g = test_graphs[idx]
    batch = Batch.from_data_list([g]).to(device)
    with torch.no_grad():
        logits, _ = model(batch)
        probs = F.softmax(logits, dim=1)[0]

    cp = g.cp.item()
    true_wdl = g.y.tolist()
    pred_wdl = probs.cpu().tolist()
    pred_val = pred_wdl[0] - pred_wdl[2]

    fen_short = f"cp={cp:.0f}"

    true_str = f"W:{true_wdl[0]:.2f} D:{true_wdl[1]:.2f} L:{true_wdl[2]:.2f}"
    pred_str = f"W:{pred_wdl[0]:.2f} D:{pred_wdl[1]:.2f} L:{pred_wdl[2]:.2f}"

    log.info(f"{fen_short:>45s} | {cp:>8.0f} | {true_str:>12s} | {pred_str:>12s} | {pred_val:>+8.3f}")

In [ ]:
# ============================================================
# Save model for download
# ============================================================

log.info(f"Best model saved at: {BEST_MODEL_PATH}")
log.info(f"Model size: {Path(BEST_MODEL_PATH).stat().st_size / 1024:.1f} KB")
log.info("")
log.info("To use this model locally for playing:")
log.info("  1. Download best_model.pt")
log.info("  2. Place it in your ChessGCN project folder")
log.info("  3. Run: python play.py --checkpoint best_model.pt")

In [ ]:
# ============================================================
# MCTS: Monte Carlo Tree Search with neural network guidance
# ============================================================

class MCTSNode:
    """A node in the MCTS search tree."""
    __slots__ = [
        "board", "parent", "move", "children",
        "visit_count", "total_value", "prior",
        "is_expanded", "is_terminal", "terminal_value",
    ]

    def __init__(self, board, parent=None, move=None, prior=0.0):
        self.board = board
        self.parent = parent
        self.move = move
        self.children = {}
        self.visit_count = 0
        self.total_value = 0.0
        self.prior = prior
        self.is_expanded = False
        self.is_terminal = board.is_game_over(claim_draw=True)
        self.terminal_value = None
        if self.is_terminal:
            result = board.result(claim_draw=True)
            if result == "1-0":
                self.terminal_value = 1.0 if board.turn == chess.WHITE else -1.0
            elif result == "0-1":
                self.terminal_value = -1.0 if board.turn == chess.WHITE else 1.0
            else:
                self.terminal_value = 0.0

    @property
    def q_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.total_value / self.visit_count

    def ucb_score(self, c_puct):
        if self.parent is None:
            return 0.0
        exploration = c_puct * self.prior * math.sqrt(self.parent.visit_count) / (1 + self.visit_count)
        return self.q_value + exploration


class MCTSEngine:
    """Monte Carlo Tree Search with neural network value + policy guidance."""

    def __init__(self, model, device, c_puct=1.5, num_simulations=128,
                 dirichlet_alpha=0.3, dirichlet_epsilon=0.25):
        self.model = model
        self.device = device
        self.c_puct = c_puct
        self.num_simulations = num_simulations
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_epsilon = dirichlet_epsilon

    @torch.no_grad()
    def _evaluate(self, board):
        fen = board.fen()
        graph = fen_to_graph(fen)
        batch = Batch.from_data_list([graph]).to(self.device)
        value_logits, policy_logits = self.model(batch)

        wdl = F.softmax(value_logits, dim=1)[0]
        value = (wdl[0] - wdl[2]).item()

        legal_moves = list(board.legal_moves)
        if policy_logits is not None and len(legal_moves) > 0:
            probs = F.softmax(policy_logits, dim=0).cpu().numpy()
            policy_dict = {move: float(probs[i]) for i, move in enumerate(legal_moves)}
        else:
            uniform = 1.0 / max(len(legal_moves), 1)
            policy_dict = {m: uniform for m in legal_moves}

        return value, policy_dict

    def _select(self, node):
        while node.is_expanded and not node.is_terminal:
            node = max(node.children.values(), key=lambda c: c.ucb_score(self.c_puct))
        return node

    def _expand(self, node):
        if node.is_terminal:
            return node.terminal_value
        value, policy_dict = self._evaluate(node.board)
        for move, prior in policy_dict.items():
            child_board = node.board.copy()
            child_board.push(move)
            node.children[move] = MCTSNode(child_board, parent=node, move=move, prior=prior)
        node.is_expanded = True
        return value

    def _backpropagate(self, node, value):
        while node is not None:
            node.visit_count += 1
            node.total_value += value
            value = -value
            node = node.parent

    def search(self, board, temperature=1.0):
        root = MCTSNode(board.copy())
        self._expand(root)

        # Add Dirichlet noise to root priors
        if root.children:
            noise = np.random.dirichlet([self.dirichlet_alpha] * len(root.children))
            for i, child in enumerate(root.children.values()):
                child.prior = (1 - self.dirichlet_epsilon) * child.prior + self.dirichlet_epsilon * noise[i]

        for _ in range(self.num_simulations):
            leaf = self._select(root)
            value = self._expand(leaf)
            self._backpropagate(leaf, value)

        move_visits = {move: child.visit_count for move, child in root.children.items()}
        total_visits = sum(move_visits.values())

        if total_visits == 0:
            n = max(len(move_visits), 1)
            return {m: 1.0 / n for m in move_visits}, root

        if temperature < 0.01:
            best_move = max(move_visits, key=move_visits.get)
            move_probs = {m: 0.0 for m in move_visits}
            move_probs[best_move] = 1.0
        else:
            visits = np.array([move_visits[m] for m in move_visits], dtype=np.float64)
            if temperature != 1.0:
                visits = visits ** (1.0 / temperature)
            probs = visits / visits.sum()
            move_probs = {move: float(probs[i]) for i, move in enumerate(move_visits)}

        return move_probs, root


log.info("MCTS engine defined.")

## 5. Self-Play Training (Optional)

Set `RUN_SELFPLAY = True` in the configuration cell to enable this section.

This implements an AlphaZero-style training loop:
1. **Policy pretraining**: Bootstrap the policy head using Stockfish best moves
2. **MCTS**: Monte Carlo Tree Search with PUCT selection, guided by both value and policy heads
3. **Self-play**: Generate training data by playing games against itself
4. **Training**: Combined value + policy loss on replay buffer
5. **Evaluation**: Gate new models by playing against the current best

In [ ]:
# ============================================================
# Self-play training pipeline
# ============================================================

class ReplayBuffer:
    """Circular buffer of training samples."""
    def __init__(self, max_size=300_000):
        self.buffer = deque(maxlen=max_size)

    def add(self, graph, policy_target, value_target):
        entry = Data(
            x=graph.x, edge_index=graph.edge_index, edge_attr=graph.edge_attr,
            policy_target=policy_target,
            value_target=torch.tensor([value_target], dtype=torch.float),
        )
        self.buffer.append(entry)

    def sample(self, batch_size):
        indices = random.sample(range(len(self.buffer)), min(batch_size, len(self.buffer)))
        return [self.buffer[i] for i in indices]

    def __len__(self):
        return len(self.buffer)


def _scatter_log_softmax(logits, batch_idx):
    """Numerically stable log-softmax grouped by batch_idx."""
    num_graphs = batch_idx.max().item() + 1
    max_vals = torch.full((num_graphs,), -1e9, device=logits.device)
    max_vals.scatter_reduce_(0, batch_idx, logits, reduce="amax")
    shifted = logits - max_vals[batch_idx]
    exp_vals = shifted.exp()
    sum_exp = torch.zeros(num_graphs, device=logits.device)
    sum_exp.scatter_add_(0, batch_idx, exp_vals)
    return shifted - sum_exp[batch_idx].log()


def per_graph_cross_entropy(logits, targets, batch_idx):
    log_probs = _scatter_log_softmax(logits, batch_idx)
    num_graphs = batch_idx.max().item() + 1
    return -(targets * log_probs).sum() / num_graphs


def play_one_game(mcts_engine, max_moves=512, temp_threshold=30,
                  resign_threshold=-0.95, resign_enabled=True):
    """Play a single self-play game."""
    board = chess.Board()
    game_data = []
    game_record = []

    for move_num in range(max_moves):
        if board.is_game_over(claim_draw=True):
            break
        temperature = 1.0 if move_num < temp_threshold else 0.01
        move_probs, root = mcts_engine.search(board, temperature=temperature)
        game_data.append((board.fen(), move_probs, board.turn))

        # Top-5 for game record
        sorted_moves = sorted(move_probs.items(), key=lambda x: x[1], reverse=True)[:5]
        top5 = []
        for move, prob in sorted_moves:
            child = root.children.get(move)
            top5.append({
                "uci": move.uci(), "san": board.san(move),
                "visits": child.visit_count if child else 0,
                "prob": round(prob, 4),
                "q": round(child.q_value, 4) if child and child.visit_count > 0 else 0.0,
            })

        if resign_enabled and root.q_value < resign_threshold:
            game_record.append({"fen": board.fen(), "side": "white" if board.turn == chess.WHITE else "black",
                                "move": "resign", "top5": top5, "root_q": round(root.q_value, 4)})
            result = -1.0 if board.turn == chess.WHITE else 1.0
            return game_data, game_record, result

        moves = list(move_probs.keys())
        probs = np.array([move_probs[m] for m in moves])
        probs = probs / probs.sum()
        chosen = np.random.choice(len(moves), p=probs)
        chosen_move = moves[chosen]

        game_record.append({
            "fen": board.fen(), "side": "white" if board.turn == chess.WHITE else "black",
            "move": chosen_move.uci(), "move_san": board.san(chosen_move),
            "top5": top5, "root_q": round(root.q_value, 4),
        })
        board.push(chosen_move)

    if board.is_game_over(claim_draw=True):
        r = board.result(claim_draw=True)
        result = 1.0 if r == "1-0" else (-1.0 if r == "0-1" else 0.0)
    else:
        result = 0.0

    return game_data, game_record, result


def add_game_to_buffer(game_data, result, replay_buffer):
    for fen, move_probs, side_to_move in game_data:
        graph = fen_to_graph(fen)
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        policy_target = torch.zeros(len(legal_moves), dtype=torch.float)
        for i, move in enumerate(legal_moves):
            policy_target[i] = move_probs.get(move, 0.0)
        if policy_target.sum() > 0:
            policy_target = policy_target / policy_target.sum()
        value = result if side_to_move == chess.WHITE else -result
        replay_buffer.add(graph, policy_target, value)


def train_on_buffer(model, replay_buffer, optimizer, device, batch_size=256, num_batches=200):
    model.train()
    total_vloss, total_ploss, n = 0.0, 0.0, 0
    for _ in range(num_batches):
        if len(replay_buffer) < batch_size:
            break
        samples = replay_buffer.sample(batch_size)
        batch = Batch.from_data_list(samples).to(device)
        optimizer.zero_grad()
        value_logits, policy_logits = model(batch)

        z = batch.value_target.view(-1)
        target_wdl = torch.stack([(z + 1) / 2, 1 - z.abs(), (1 - z) / 2], dim=1)
        log_probs = F.log_softmax(value_logits, dim=1)
        value_loss = F.kl_div(log_probs, target_wdl, reduction="batchmean")

        if policy_logits is not None and policy_logits.numel() > 0:
            edge_batch = batch.batch[batch.edge_index[0]]
            policy_loss = per_graph_cross_entropy(policy_logits, batch.policy_target, edge_batch)
        else:
            policy_loss = torch.tensor(0.0, device=device)

        loss = value_loss + policy_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_vloss += value_loss.item()
        total_ploss += policy_loss.item()
        n += 1
    return {"value_loss": total_vloss / max(n, 1), "policy_loss": total_ploss / max(n, 1)}


def evaluate_models(challenger, best_model, device, num_games=20, num_simulations=32, max_moves=200):
    challenger.eval()
    best_model.eval()
    c_mcts = MCTSEngine(challenger, device, num_simulations=num_simulations)
    b_mcts = MCTSEngine(best_model, device, num_simulations=num_simulations)
    challenger_points = 0.0
    for game_idx in range(num_games):
        challenger_is_white = (game_idx % 2 == 0)
        white_mcts = c_mcts if challenger_is_white else b_mcts
        black_mcts = b_mcts if challenger_is_white else c_mcts
        board = chess.Board()
        for _ in range(max_moves):
            if board.is_game_over(claim_draw=True):
                break
            current = white_mcts if board.turn == chess.WHITE else black_mcts
            move_probs, _ = current.search(board, temperature=0.01)
            best_move = max(move_probs, key=move_probs.get)
            board.push(best_move)
        r = board.result(claim_draw=True)
        if r == "1-0":
            challenger_points += 1.0 if challenger_is_white else 0.0
        elif r == "0-1":
            challenger_points += 0.0 if challenger_is_white else 1.0
        else:
            challenger_points += 0.5
    return challenger_points / num_games


log.info("Self-play functions defined.")

In [ ]:
# ============================================================
# Run self-play training (only if RUN_SELFPLAY = True)
# ============================================================

if not RUN_SELFPLAY:
    log.info("Self-play training SKIPPED (set RUN_SELFPLAY = True to enable)")
else:
    # Load the best supervised model as starting point
    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True, map_location=device))
    log.info("Loaded supervised checkpoint for self-play bootstrap")

    # ── Phase 1: Policy pretraining ──
    if PRETRAIN_POLICY:
        # Load data (with optional stratified sampling)
        pretrain_df = pd.read_csv(DATA_PATH)
        pretrain_df["cp"] = pretrain_df["Evaluation"].apply(parse_evaluation)
        if PRETRAIN_SAMPLES is not None and PRETRAIN_SAMPLES < len(pretrain_df):
            pretrain_df = load_and_sample(pretrain_df, PRETRAIN_SAMPLES)
        log.info(f"=== Policy Pretraining ({PRETRAIN_EPOCHS} epochs on {len(pretrain_df):,} samples) ===")

        # Build graphs with policy targets (best move = 1.0)
        policy_graphs = []
        skipped = 0
        for _, row in tqdm(pretrain_df.iterrows(), total=len(pretrain_df), desc="Building policy dataset"):
            fen = row["Position"]
            wdl = cp_to_wdl(row["cp"])
            best_move_uci = str(row["Best Move"]).strip()
            graph = fen_to_graph(fen, wdl)
            board = chess.Board(fen)
            legal_moves = list(board.legal_moves)
            target_idx = -1
            for i, move in enumerate(legal_moves):
                if move.uci() == best_move_uci:
                    target_idx = i
                    break
            if target_idx == -1:
                skipped += 1
                continue
            policy_target = torch.zeros(len(legal_moves), dtype=torch.float)
            policy_target[target_idx] = 1.0
            graph.policy_target = policy_target
            policy_graphs.append(graph)

        log.info(f"Built {len(policy_graphs)} policy graphs ({skipped} skipped)")

        ptrain_g, pval_g = train_test_split(policy_graphs, test_size=0.1, random_state=42)
        ptrain_loader = DataLoader(ptrain_g, batch_size=BATCH_SIZE, shuffle=True)
        pval_loader = DataLoader(pval_g, batch_size=BATCH_SIZE)

        # Differential LR
        backbone_params = [p for n, p in model.named_parameters() if "policy_mlp" not in n]
        policy_params = [p for n, p in model.named_parameters() if "policy_mlp" in n]
        pretrain_opt = AdamW([
            {"params": backbone_params, "lr": SELFPLAY_LR * 0.1},
            {"params": policy_params, "lr": SELFPLAY_LR * 10},
        ], weight_decay=1e-4)

        for epoch in range(1, PRETRAIN_EPOCHS + 1):
            model.train()
            t_vloss, t_ploss, batches = 0.0, 0.0, 0
            for batch in ptrain_loader:
                batch = batch.to(device)
                pretrain_opt.zero_grad()
                value_logits, policy_logits = model(batch)
                target_wdl = batch.y.view(-1, 3)
                vloss = F.kl_div(F.log_softmax(value_logits, dim=1), target_wdl, reduction="batchmean")
                if policy_logits is not None and policy_logits.numel() > 0:
                    edge_batch = batch.batch[batch.edge_index[0]]
                    ploss = per_graph_cross_entropy(policy_logits, batch.policy_target, edge_batch)
                else:
                    ploss = torch.tensor(0.0, device=device)
                loss = vloss + ploss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                pretrain_opt.step()
                t_vloss += vloss.item()
                t_ploss += ploss.item()
                batches += 1
            log.info(f"  Pretrain {epoch}/{PRETRAIN_EPOCHS} | val_loss={t_vloss/batches:.4f} pol_loss={t_ploss/batches:.4f}")

        torch.save(model.state_dict(), "pretrained_dual_head.pt")
        log.info("Policy pretraining complete")

    # ── Phase 2: Self-play loop ──
    log.info(f"=== Self-Play Training ({SELFPLAY_ITERATIONS} iterations) ===")
    best_model = copy.deepcopy(model)
    best_model.eval()
    replay_buffer = ReplayBuffer(max_size=SELFPLAY_BUFFER_SIZE)
    sp_optimizer = AdamW(model.parameters(), lr=SELFPLAY_LR, weight_decay=1e-4)
    games_log = []

    for iteration in range(1, SELFPLAY_ITERATIONS + 1):
        iter_start = time.time()
        log.info(f"--- Iteration {iteration}/{SELFPLAY_ITERATIONS} ---")

        # Self-play
        model.eval()
        mcts_engine = MCTSEngine(model, device, num_simulations=SELFPLAY_SIMULATIONS)
        outcomes = {"W": 0, "D": 0, "B": 0}
        positions_added = 0

        for g in range(SELFPLAY_GAMES_PER_ITER):
            resign_on = random.random() > 0.2
            game_data, game_record, result = play_one_game(
                mcts_engine, max_moves=SELFPLAY_MAX_MOVES, resign_enabled=resign_on,
            )
            add_game_to_buffer(game_data, result, replay_buffer)
            positions_added += len(game_data)
            result_str = {1.0: "1-0", -1.0: "0-1", 0.0: "1/2-1/2"}.get(result, "?")
            games_log.append({"iteration": iteration, "game": g + 1, "result": result_str,
                              "num_moves": len(game_record), "moves": game_record})
            outcome_key = {1.0: "W", -1.0: "B", 0.0: "D"}.get(result, "?")
            outcomes[outcome_key] = outcomes.get(outcome_key, 0) + 1

        log.info(f"  Games: W:{outcomes['W']} D:{outcomes['D']} B:{outcomes['B']} | "
                 f"{positions_added} positions | Buffer: {len(replay_buffer):,}")

        # Training — scale steps to avoid overfitting on small buffer
        actual_steps = min(SELFPLAY_TRAIN_STEPS, len(replay_buffer) // BATCH_SIZE)
        model.train()
        losses = train_on_buffer(model, replay_buffer, sp_optimizer, device,
                                 batch_size=BATCH_SIZE, num_batches=actual_steps)
        log.info(f"  Training ({actual_steps} steps): val_loss={losses['value_loss']:.4f} pol_loss={losses['policy_loss']:.4f}")

        # Save checkpoint every iteration (crash resilience)
        torch.save(model.state_dict(), "selfplay_latest.pt")
        if iteration % 5 == 0:
            torch.save(model.state_dict(), f"selfplay_iter{iteration}.pt")
        log.info(f"  Checkpoint saved (iter {iteration})")

        # Evaluation
        if iteration % SELFPLAY_EVAL_INTERVAL == 0:
            model.eval()
            win_rate = evaluate_models(model, best_model, device,
                                       num_games=SELFPLAY_EVAL_GAMES,
                                       num_simulations=SELFPLAY_EVAL_SIMS,
                                       max_moves=SELFPLAY_MAX_MOVES)
            log.info(f"  Eval: challenger win rate = {win_rate:.1%}")
            if win_rate >= 0.55:
                log.info(f"  New best model!")
                best_model = copy.deepcopy(model)
                best_model.eval()
                torch.save(model.state_dict(), f"selfplay_best_iter{iteration}.pt")

        log.info(f"  Time: {time.time() - iter_start:.0f}s")

    torch.save(model.state_dict(), "selfplay_final.pt")
    log.info("Self-play training complete!")

    # Save game log
    with open("selfplay_games.jsonl", "w") as f:
        for entry in games_log:
            f.write(json.dumps(entry) + "\n")
    log.info(f"Saved {len(games_log)} game records to selfplay_games.jsonl")